# Comparação de variantes do pipeline no conjunto de teste

## Objetivo

Compara threshold normal/fuzzy e segmentação RG/FC, incluindo métricas, pares de exames e exemplos qualitativos 3D.

In [ ]:
from pathlib import Path
import copy
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from utils.project.notebook_env import configure_notebook_environment
    REPO_ROOT = configure_notebook_environment(chdir_to_src=False)
except Exception:
    current = Path.cwd().resolve()
    REPO_ROOT = next(
        path for path in [current, *current.parents]
        if (path / "src").exists() and (path / "output").exists()
    )

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils.project.config import load_config_json
from utils.project.notebook_env import (
    resolve_imagecas_base_path,
    resolve_processed_imagecas_path,
)
from utils.experiments import run_qualitative_pipeline_case
from utils.visualization.volume import (
    visualize_aorta_ostia_artery,
)

from utils.visualization.variant_comparison import (
    best_variant_by_suffix,
    build_dice_stats_by_variant,
    build_pair_outcome_counts,
    build_delta_summary_vs_reference,
    build_ranking_table,
    largest_pair_changes,
    load_variant_results,
    make_pair_delta,
    pair_summary,
    plot_largest_pair_changes,
    plot_ostia_status_by_variant,
    select_qualitative_pair_cases,
)

RESULT_ROOT = REPO_ROOT / "output/segmentation/runs/mid_res/fuzzy_comparison"
ANALYSIS_DIR = REPO_ROOT / "output/segmentation/analysis/threshold_pipeline_comparison"
TABLE_DIR = ANALYSIS_DIR / "tables"
FIGURE_DIR = ANALYSIS_DIR / "figures"
QUALITATIVE_DIR = ANALYSIS_DIR / "qualitative_3d"
for directory in (TABLE_DIR, FIGURE_DIR, QUALITATIVE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)


def save_current_figure(name: str):
    path = FIGURE_DIR / name
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Figura salva em: {path.relative_to(REPO_ROOT)}")

RESULT_ROOT


## Configuração

Ajuste nesta seção apenas os parâmetros da análise; o pipeline base não é alterado.

## Carregamento dos Resultados

A análise usa o `ostios_test_summary.csv` de cada variante como fonte principal, porque esse CSV registra por imagem o método de threshold e o método arterial efetivamente usados.

In [ ]:
PREFERRED_ORDER = [
    "normal_rg",
    "th_fuzzy_rg",
    "normal_fc",
    "th_fuzzy_fc",
]

PRETTY_NAMES = {
    "normal_rg": "Normal + RG",
    "th_fuzzy_rg": "Fuzzy threshold + RG",
    "normal_fc": "Normal + FC",
    "th_fuzzy_fc": "Fuzzy threshold + FC",
}

In [ ]:
results_df, summary_df = load_variant_results(
    RESULT_ROOT,
    split="test",
    preferred_order=PREFERRED_ORDER,
    pretty_names=PRETTY_NAMES,
    repo_root=REPO_ROOT,
)

print(f"Runs carregados: {summary_df.shape[0]}")
print(f"Linhas por imagem: {results_df.shape[0]}")

In [ ]:
display(summary_df)

## Análise

As subseções abaixo apresentam as métricas, tabelas ou visualizações do objetivo definido.

## Resumo Geral

A tabela abaixo ordena as variantes por sucesso dos óstios e Dice médio. O sucesso dos óstios considera `both correct` ou `both tolerable` como sucesso.

In [ ]:
ranking_df = summary_df.sort_values(
    ["ostia_success_rate", "mean_dice", "mean_dice_success_ostia"],
    ascending=False,
).copy()
ranking_display = build_ranking_table(summary_df)

for col in ["ostia_detected_rate", "ostia_success_rate"]:
    if col in ranking_display.columns:
        ranking_display[col] = 100 * ranking_display[col]

In [ ]:
display(ranking_display.round({
    "ostia_detected_rate": 1,
    "ostia_success_rate": 1,
    "mean_dice": 4,
    "median_dice": 4,
    "mean_dice_success_ostia": 4,
}))

## Detecção dos Óstios

Aqui o status é separado em: ambos corretos, ambos toleráveis, encontrados porém incorretos e não encontrado/erro.

In [ ]:
ostia_plot = summary_df.copy()

In [ ]:
plot_ostia_status_by_variant(
    ostia_plot,
    preferred_order=PREFERRED_ORDER,
    pretty_names=PRETTY_NAMES,
    save_path=FIGURE_DIR / "ostia_status_by_variant.png",
)
print(f"Figura salva em: {(FIGURE_DIR / 'ostia_status_by_variant.png').relative_to(REPO_ROOT)}")
plt.show()

## Dice Score por Variante

A tabela abaixo resume o Dice arterial por variante. O gráfico em seguida mostra média e mediana para comparação visual.

In [ ]:
dice_plot = summary_df.copy()
dice_plot["variant_label"] = pd.Categorical(
    dice_plot["variant_label"],
    [PRETTY_NAMES.get(name, name) for name in PREFERRED_ORDER],
    ordered=True,
)
dice_plot = dice_plot.sort_values("variant_label")

In [ ]:
dice_stats_df = build_dice_stats_by_variant(results_df, PREFERRED_ORDER)

dice_stats_csv_path = TABLE_DIR / "dice_stats_by_variant.csv"
dice_stats_df.to_csv(dice_stats_csv_path, index=False)
print(f"CSV salvo em: {dice_stats_csv_path.relative_to(REPO_ROOT)}")

display(dice_stats_df.round({
    "mean_dice": 4,
    "max_dice": 4,
    "min_dice": 4,
    "std_dice": 4,
    "median_dice": 4,
}))

In [ ]:
x = np.arange(len(dice_plot))
width = 0.38
fig, ax = plt.subplots(figsize=(12, 5))
mean_bars = ax.bar(x - width/2, dice_plot["mean_dice"], width, label="Média", color="#4c78a8")
median_bars = ax.bar(x + width/2, dice_plot["median_dice"], width, label="Mediana", color="#f58518")

for bars in [mean_bars, median_bars]:
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.008,
            f"{height:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

ax.set_xticks(x)
ax.set_xticklabels(dice_plot["variant_label"].astype(str), rotation=35, ha="right")
ax.set_ylabel("Dice arterial", fontsize=12)
y_max = max(0.75, float(dice_plot[["mean_dice", "median_dice"]].max().max()) + 0.08)
ax.set_ylim(0, y_max)
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("dice_mean_median_by_variant.png")
plt.show()

## Comparação com Baseline

A célula abaixo usa `Normal + RG` como baseline e calcula o ganho/perda de Dice por imagem.

In [ ]:
baseline_variant = "normal_rg"
delta_summary_df = build_delta_summary_vs_reference(
    results_df,
    baseline_variant,
    variants=PREFERRED_ORDER,
    pretty_names=PRETTY_NAMES,
)

In [ ]:
display(delta_summary_df.round({
    "mean_delta": 4,
    "median_delta": 4,
    "max_gain": 4,
    "max_loss": 4,
}))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_delta = delta_summary_df.copy()
colors = ["#2ca02c" if value >= 0 else "#d62728" for value in plot_delta["mean_delta"]]
bars = ax.bar(plot_delta["variant_label"], plot_delta["mean_delta"], color=colors)

for bar, value in zip(bars, plot_delta["mean_delta"]):
    va = "bottom" if value >= 0 else "top"
    offset = 0.002 if value >= 0 else -0.002
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + offset,
        f"{value:+.4f}",
        ha="center",
        va=va,
        fontsize=9,
    )

ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Delta médio de Dice vs Normal + RG", fontsize=12)
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=35, labelsize=10)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("dice_delta_vs_normal_rg.png")
plt.show()

## Maiores Variações por Comparação

As próximas células comparam pares específicos de variantes. Em cada comparação, o delta é calculado como `comparação - referência`. Valores positivos indicam que a segunda variante melhorou o Dice em relação à primeira.

In [ ]:
PAIR_COMPARISONS = None
print("Os pares são gerados automaticamente a partir das variantes carregadas.")

### Resumo Numérico das Variações

Legenda das variantes: `C1 = Normal + RG`, `C2 = Fuzzy threshold + RG`, `C3 = Normal + FC`, `C4 = Fuzzy threshold + FC`.

A tabela abaixo mostra, para cada comparação, quantos exames melhoraram, pioraram ou ficaram com o mesmo Dice em relação à variante de referência. O delta é calculado como `comparação - referência`.


In [ ]:
pair_outcome_counts_df = build_pair_outcome_counts(
    results_df,
    PAIR_COMPARISONS,
    pretty_names=PRETTY_NAMES,
    min_delta=0.02,
    variant_order=PREFERRED_ORDER,
)

pair_outcome_counts_path = TABLE_DIR / "pair_outcome_counts.csv"
pair_outcome_counts_df.to_csv(pair_outcome_counts_path, index=False)

compact_pair_outcome_counts_df = pair_outcome_counts_df[
    [
        "comparison_name",
        "n_images",
        "comparison_better_n",
        "reference_better_n",
        "same_dice_n",
        "mean_delta",
    ]
]

display(compact_pair_outcome_counts_df.round({
    "mean_delta": 4,
}))
print(f"CSV salvo em: {pair_outcome_counts_path.relative_to(REPO_ROOT)}")

### Threshold fuzzy vs normal, ambos com RG

In [ ]:
display(pair_summary(results_df, "normal_rg", "th_fuzzy_rg", PRETTY_NAMES).round({"mean_delta": 4, "median_delta": 4, "max_gain": 4, "max_loss": 4}))

In [ ]:
display(largest_pair_changes(results_df, "normal_rg", "th_fuzzy_rg", top_n=10).round({"reference_dice": 4, "comparison_dice": 4, "dice_delta": 4}))

In [ ]:
plot_largest_pair_changes(
    results_df,
    "normal_rg",
    "th_fuzzy_rg",
    title="Threshold fuzzy vs normal, ambos com RG",
    save_path=FIGURE_DIR / "largest_changes_th_fuzzy_rg_vs_normal_rg.png",
    top_n=10,
)
print(f"Figura salva em: {(FIGURE_DIR / 'largest_changes_th_fuzzy_rg_vs_normal_rg.png').relative_to(REPO_ROOT)}")
plt.show()

### Threshold fuzzy vs normal, ambos com FC

In [ ]:
display(pair_summary(results_df, "normal_fc", "th_fuzzy_fc", PRETTY_NAMES).round({"mean_delta": 4, "median_delta": 4, "max_gain": 4, "max_loss": 4}))

In [ ]:
display(largest_pair_changes(results_df, "normal_fc", "th_fuzzy_fc", top_n=10).round({"reference_dice": 4, "comparison_dice": 4, "dice_delta": 4}))

In [ ]:
plot_largest_pair_changes(
    results_df,
    "normal_fc",
    "th_fuzzy_fc",
    title="Threshold fuzzy vs normal, ambos com FC",
    save_path=FIGURE_DIR / "largest_changes_th_fuzzy_fc_vs_normal_fc.png",
    top_n=10,
)
print(f"Figura salva em: {(FIGURE_DIR / 'largest_changes_th_fuzzy_fc_vs_normal_fc.png').relative_to(REPO_ROOT)}")
plt.show()

### FC vs RG com threshold normal

In [ ]:
display(pair_summary(results_df, "normal_rg", "normal_fc", PRETTY_NAMES).round({"mean_delta": 4, "median_delta": 4, "max_gain": 4, "max_loss": 4}))

In [ ]:
display(largest_pair_changes(results_df, "normal_rg", "normal_fc", top_n=15).round({"reference_dice": 4, "comparison_dice": 4, "dice_delta": 4}))

In [ ]:
plot_largest_pair_changes(
    results_df,
    "normal_rg",
    "normal_fc",
    title="FC vs RG com threshold normal",
    save_path=FIGURE_DIR / "largest_changes_normal_fc_vs_normal_rg.png",
    top_n=15,
)
print(f"Figura salva em: {(FIGURE_DIR / 'largest_changes_normal_fc_vs_normal_rg.png').relative_to(REPO_ROOT)}")
plt.show()

### FC vs RG com threshold fuzzy

In [ ]:
display(pair_summary(results_df, "th_fuzzy_rg", "th_fuzzy_fc", PRETTY_NAMES).round({"mean_delta": 4, "median_delta": 4, "max_gain": 4, "max_loss": 4}))

In [ ]:
display(largest_pair_changes(results_df, "th_fuzzy_rg", "th_fuzzy_fc", top_n=15).round({"reference_dice": 4, "comparison_dice": 4, "dice_delta": 4}))

In [ ]:
plot_largest_pair_changes(
    results_df,
    "th_fuzzy_rg",
    "th_fuzzy_fc",
    title="FC vs RG com threshold fuzzy",
    save_path=FIGURE_DIR / "largest_changes_th_fuzzy_fc_vs_th_fuzzy_rg.png",
    top_n=15,
)
print(f"Figura salva em: {(FIGURE_DIR / 'largest_changes_th_fuzzy_fc_vs_th_fuzzy_rg.png').relative_to(REPO_ROOT)}")
plt.show()

## Melhor RG vs Melhor FC

Esta comparação seleciona automaticamente a melhor variante RG e a melhor variante FC pelo ranking geral e mostra onde o FC mais ganha ou perde em relação ao RG.

In [ ]:
best_rg_variant = best_variant_by_suffix(ranking_df, "_rg")
best_fc_variant = best_variant_by_suffix(ranking_df, "_fc")
print(f"Melhor RG: {PRETTY_NAMES.get(best_rg_variant, best_rg_variant)}")
print(f"Melhor FC: {PRETTY_NAMES.get(best_fc_variant, best_fc_variant)}")

In [ ]:
display(pair_summary(results_df, best_rg_variant, best_fc_variant, PRETTY_NAMES).round({"mean_delta": 4, "median_delta": 4, "max_gain": 4, "max_loss": 4}))

In [ ]:
display(largest_pair_changes(results_df, best_rg_variant, best_fc_variant, top_n=10).round({"reference_dice": 4, "comparison_dice": 4, "dice_delta": 4}))

In [ ]:
plot_largest_pair_changes(
    results_df,
    best_rg_variant,
    best_fc_variant,
    title="Melhor FC vs melhor RG",
    save_path=FIGURE_DIR / "largest_changes_best_fc_vs_best_rg.png",
    top_n=10,
)
print(f"Figura salva em: {(FIGURE_DIR / 'largest_changes_best_fc_vs_best_rg.png').relative_to(REPO_ROOT)}")
plt.show()

## Comparação Qualitativa 3D

Esta seção compara o pipeline original (`normal_rg`) com a melhor variante encontrada no ranking. São selecionados três casos: um com status equivalente dos óstios, um com status diferente dos óstios sem Dice muito baixo nas duas máscaras, e um caso em que os Dice das duas variantes ficam próximos das médias de suas respectivas variantes. Para o status equivalente, `both correct` e `both tolerable` são tratados como o mesmo grupo de sucesso.


In [ ]:

try:
    QUAL_BASE_PATH = resolve_imagecas_base_path()
except FileNotFoundError as exc:
    QUAL_BASE_PATH = None
    print(f"Dataset ImageCAS não encontrado para a análise qualitativa: {exc}")

try:
    QUAL_SAVE_PATH = resolve_processed_imagecas_path()
except FileNotFoundError:
    QUAL_SAVE_PATH = QUALITATIVE_DIR / "cache_lookup"
    QUAL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

QUAL_REFERENCE_VARIANT = "normal_rg"
ranked_variants = ranking_df["folder_variant"].astype(str).tolist()
QUAL_COMPARISON_VARIANT = next(
    variant for variant in ranked_variants
    if variant != QUAL_REFERENCE_VARIANT
)
print("Comparação qualitativa:")
print(f"  Referência: {PRETTY_NAMES.get(QUAL_REFERENCE_VARIANT, QUAL_REFERENCE_VARIANT)}")
print(f"  Variante:   {PRETTY_NAMES.get(QUAL_COMPARISON_VARIANT, QUAL_COMPARISON_VARIANT)}")


In [ ]:
QUALITATIVE_CASES, QUALITATIVE_CASES_DF = select_qualitative_pair_cases(
    results_df,
    QUAL_REFERENCE_VARIANT,
    QUAL_COMPARISON_VARIANT,
    min_dice=0.02,
    same_status_min_delta=0.0,
    same_status_max_delta=0.20,
    same_status_require_both_correct=True,
    different_status_min_delta=0.0,
    different_status_max_delta=0.30,
    different_reference_status_group="wrong",
    different_reference_one_ostium_correct=True,
    different_comparison_success=True,
    excluded_img_ids_by_case={
        "same_ostia": [70, 768],
        "different_ostia": [574, 610],
    },
)
QUALITATIVE_CASES_PATH = QUALITATIVE_DIR / "selected_qualitative_cases.csv"
QUALITATIVE_CASES_DF.to_csv(QUALITATIVE_CASES_PATH, index=False)

display(QUALITATIVE_CASES_DF[[
    "case_key",
    "case_label",
    "IMG_ID",
    "reference_dice",
    "comparison_dice",
    "dice_delta",
    "same_ostia_status_group",
    "reference_status_group",
    "comparison_status_group",
    "same_ostia_points",
    "reference_one_ostium_correct",
    "comparison_ostia_success",
    "reference_both_ostia_correct_bool",
    "reference_both_ostia_tolerable_bool",
    "comparison_both_ostia_correct_bool",
    "comparison_both_ostia_tolerable_bool",
    "reference_mean_dice",
    "comparison_mean_dice",
    "reference_distance_to_mean",
    "comparison_distance_to_mean",
    "reference_ostia_status",
    "comparison_ostia_status",
    "reference_left_ostium",
    "comparison_left_ostium",
    "reference_right_ostium",
    "comparison_right_ostium",
]].round({
    "reference_dice": 4,
    "comparison_dice": 4,
    "dice_delta": 4,
    "reference_mean_dice": 4,
    "comparison_mean_dice": 4,
    "reference_distance_to_mean": 4,
    "comparison_distance_to_mean": 4,
}))
print(f"CSV salvo em: {QUALITATIVE_CASES_PATH.relative_to(REPO_ROOT)}")


In [ ]:
QUALITATIVE_CACHE = {}


def load_qualitative_variant_config(variant: str) -> tuple[dict, Path]:
    run_dir = summary_df.loc[summary_df["folder_variant"] == variant, "run_dir"].iloc[0]
    config_path = REPO_ROOT / run_dir / "config/effective_pipeline_config.json"
    config = copy.deepcopy(load_config_json(str(config_path), {}))
    config["LOAD_CACHE"] = True
    config["SAVE_CACHE"] = False
    return config, config_path


def run_qualitative_pipeline(img_id: int, variant: str) -> dict:
    if QUAL_BASE_PATH is None:
        raise FileNotFoundError("Defina IMAGECAS_BASE_PATH para gerar os casos 3D.")
    cache_key = (int(img_id), variant)
    if cache_key not in QUALITATIVE_CACHE:
        config, config_path = load_qualitative_variant_config(variant)
        result = run_qualitative_pipeline_case(
            img_id,
            config,
            QUAL_BASE_PATH,
            QUAL_SAVE_PATH / variant,
            load_cache=True,
            save_cache=False,
        )
        result.update(
            {
                "variant": variant,
                "variant_label": PRETTY_NAMES.get(variant, variant),
                "config_path": config_path,
            }
        )
        QUALITATIVE_CACHE[cache_key] = result
    return QUALITATIVE_CACHE[cache_key]


def qualitative_case_row(case_key: str) -> dict:
    return QUALITATIVE_CASES[case_key]


def display_qualitative_variant(case_key: str, variant: str) -> None:
    case = qualitative_case_row(case_key)
    img_id = int(case["IMG_ID"])
    data = run_qualitative_pipeline(img_id, variant)
    metrics = results_df.loc[
        (results_df["IMG_ID"] == img_id) & (results_df["folder_variant"] == variant),
        ["variant_label", "artery_dice", "ostia_detection_status", "left_ostium", "right_ostium", "artery_voxel_count"],
    ]
    display(metrics.round({"artery_dice": 4}))
    html_path = QUALITATIVE_DIR / f"{case_key}_{variant}_img_{img_id}_aorta_ostia_artery.html"
    visualize_aorta_ostia_artery(
        data["aorta_mask"],
        data["ostia_left"],
        data["ostia_right"],
        artery_mask=data["artery_mask"],
        label_artery=data["label_artery"],
        spacing=data["scaled_spacing"],
        use_physical_coords=True,
        save_html_path=html_path,
        plot_name=f"{data['variant_label']} | IMG {img_id} | {case['case_label']}",
    )
    print(f"HTML salvo em: {html_path.relative_to(REPO_ROOT)}")


### Caso com Status de Óstios Iguais

`both correct` e `both tolerable` são considerados o mesmo grupo de sucesso nesta seleção.


In [ ]:
print("Pipeline original")
display_qualitative_variant("same_ostia", QUAL_REFERENCE_VARIANT)


In [ ]:
print("Melhor variante")
display_qualitative_variant("same_ostia", QUAL_COMPARISON_VARIANT)


### Caso com Status de Óstios Diferentes


In [ ]:
print("Pipeline original")
display_qualitative_variant("different_ostia", QUAL_REFERENCE_VARIANT)


In [ ]:
print("Melhor variante")
display_qualitative_variant("different_ostia", QUAL_COMPARISON_VARIANT)


### Caso Próximo das Médias


In [ ]:
print("Pipeline original")
display_qualitative_variant("near_mean", QUAL_REFERENCE_VARIANT)


In [ ]:
print("Melhor variante")
display_qualitative_variant("near_mean", QUAL_COMPARISON_VARIANT)


## Conclusão

O ranking, as comparações pareadas e os casos 3D permitem escolher a variante com melhor desempenho sem perder de vista falhas específicas por exame.